# Expense Tracker App - Exploratory Data Analysis

**Author:** [Harsha K S](https://github.com/harshaks17-ai)

This notebook walks through the same five stages the CLI pipeline runs, one decision at a time:

```
raw export  ->  clean  ->  aggregate  ->  analyse  ->  read the numbers
```

The dataset is **synthetic**: 50 users, 10 categories, 12 months (2025-10-01 → 2026-09-30), seed 42. It is deliberately dirty, so the cleaning section has real work to do and can be *measured* rather than assumed.

## 1. Setup

Everything lives in the `src/` package, so the notebook and the CLI share the exact same code — there is no "notebook version" of the analysis that can drift away from the shipped one.

In [ ]:
import sys
from pathlib import Path

# a Windows console may still be cp1252 and cannot encode the currency symbol
if hasattr(sys.stdout, "reconfigure"):
    sys.stdout.reconfigure(encoding="utf-8", errors="replace")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# works whether the notebook is run from notebooks/ or the project root
sys.path.insert(0, str(Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()))

from src import config, analysis, data_cleaning, data_generator, insights, utils, visualization

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 170)
pd.set_option("display.float_format", lambda v: f"{v:,.2f}")

utils.apply_matplotlib_theme()
print(f"Project: {config.PROJECT_NAME} v{config.PROJECT_VERSION}  |  seed {config.RANDOM_SEED}")
print(f"Window : {config.WINDOW_START} -> {config.WINDOW_END}  ({config.N_MONTHS} months)")
print(f"Budget : {config.CURRENCY_SYMBOL}{sum(config.CATEGORY_BUDGETS.values()):,.0f} per user per month")

## 2. Load the raw export

This is the file "as exported" — before anything has been fixed.

In [ ]:
raw = pd.read_csv(config.RAW_CSV_PATH, dtype="string")

print(f"Raw shape: {raw.shape[0]:,} rows x {raw.shape[1]} columns")
print(f"Columns  : {list(raw.columns)}")
raw.head(8)

### 2.1 What is wrong with it?

Before cleaning anything, measure the damage. This is the step most people skip: if you cannot say how dirty the file was, you cannot say how much cleaning achieved.

In [ ]:
amounts = pd.to_numeric(
    raw["amount"].astype("string").str.replace(r"[^0-9.\-]", "", regex=True), errors="coerce"
)

quality_checks = {
    "Rows": len(raw),
    "Exact duplicate rows": int(raw.duplicated().sum()),
    "Blank amounts": int(raw["amount"].isna().sum()),
    "Blank categories": int(raw["category"].isna().sum()),
    "Amounts stored as text (Rs., commas)": int(raw["amount"].astype("string").str.contains("Rs", na=False).sum()),
    "Zero or negative amounts": int((amounts <= 0).sum()),
    "Impossible outliers (>Rs.100k)": int((amounts > 100_000).sum()),
    "Categories not in the canonical list": int((~raw["category"].isin(config.CATEGORY_BUDGETS)).sum()),
    "Rows with stray whitespace": int(
        raw[["user_id", "category", "merchant", "payment_method"]]
        .apply(lambda s: s.str.len().ne(s.str.strip().str.len()), axis=0)
        .sum()
        .sum()
        if hasattr(raw["user_id"], "str")
        else 0
    ),
    "Dates not in strict ISO format": int((~raw["date"].astype(str).str.match(r"^\d{4}-\d{2}-\d{2}")).sum()),
}

for name, value in quality_checks.items():
    print(f"{name:>38}: {value:,}")

print(f"\nDirty rows as a share of the file: {raw.duplicated().sum() / len(raw) * 100:.1f}% duplicates alone.")

None of this is accident. `src/data_generator.py` injects a **known** set of defects, which is what makes cleaning measurable: the pipeline's counters can be compared against ground truth instead of being taken on faith.

| Injected defect | Why it is realistic |
| --- | --- |
| Exact duplicate exports | the same statement downloaded twice and concatenated |
| `Rs. 1,299` text amounts | exports that format currency for display |
| Blank amounts / categories | manual entries, partial rows |
| Unknown category tokens | free-text or renamed categories drifting over time |
| Casing drift | `food & dining`, `FOOD & DINING` |
| Zero / negative / ₹100k+ values | refunds, typos, misplaced decimals |
| Future-dated rows | wrong year typed in |
| Mixed date formats | a tool that changed its export format mid-year |

## 3. Clean

The cleaning function returns the cleaned frame **and** a report of what it removed — the counters are the evidence.

In [ ]:
clean, quality = data_cleaning.clean_expenses(raw)

print("--- Data quality summary ---")
for key, value in quality.items():
    if key.startswith("date_") or key in {"rows_in", "rows_out", "retention_pct", "users", "categories", "total_spend"}:
        print(f"{key:>40}: {value:,}" if isinstance(value, (int, float)) else f"{key:>40}: {value}")

In [ ]:
print("--- What was removed ---")
removals = {k: v for k, v in quality.items() if k.endswith(("_removed", "_fixed", "_parsed")) and v}
for name, value in sorted(removals.items(), key=lambda kv: -kv[1]):
    print(f"{name:>40}: {value:,}")

print(f"\nRetention: {quality['rows_out']:,} of {quality['rows_in']:,} rows kept ({quality['retention_pct']:.2f}%)")

In [ ]:
clean.head(10)

In [ ]:
# the cleaned frame must obey its own contract - these are the same assertions the test suite makes
assert clean["amount"].gt(0).all()
assert clean["amount"].le(100_000).all()
assert set(clean["category"].unique()) <= set(config.CATEGORY_BUDGETS)
assert clean["year_month"].nunique() == config.N_MONTHS
assert not clean.duplicated(subset=["user_id", "date", "category", "amount", "payment_method"]).any()
print("Cleaned frame passes every contract check.")

## 4. Headline numbers

`analysis.overview()` is the single source of truth for the KPI cards in the dashboard, so the notebook and the app can never disagree.

In [ ]:
income = data_cleaning.load_income()
o = analysis.overview(clean, income)

for key, value in o.items():
    print(f"{key:>32}: {value}")

## 5. Category mix and the Pareto cut

Budgets in this project are defined **per user per month**. Everything compared against them is therefore normalised to a per-user figure — the single most important detail in this notebook, and the source of the project's worst bug (see section 6).

In [ ]:
cat = analysis.category_summary(clean)
cat

**Reading this table:** `share_pct` answers "where does the money go", but `cumulative_pct` answers the more useful question — "how few categories explain most of it".

In [ ]:
pareto = analysis.pareto_cut(cat)
cut_share = pareto["cumulative_pct"].iloc[-1]

print(f"{len(pareto)} of {len(cat)} categories carry {cut_share:.1f}% of all spend:")
for _, row in pareto.iterrows():
    print(f"  {int(row['rank'])}. {row['category']:<18} {row['share_pct']:>5.1f}%   cumulative {row['cumulative_pct']:>5.1f}%")

print("\nA budget review should start with these and nowhere else.")

In [ ]:
fig, ax = plt.subplots(figsize=(11, 5))
order = cat.sort_values("total_spend")
ax.barh(order["category"], order["total_spend"] / 1e5, color=config.CATEGORY_COLORS[: len(order)])
ax.set_xlabel("Total spend (lakh)")
ax.set_title("Where the money actually goes - all 50 users, 12 months")
for i, (value, share) in enumerate(zip(order["total_spend"] / 1e5, order["share_pct"])):
    ax.text(value + 2, i, f"{share:.1f}%", va="center", fontsize=9)
plt.tight_layout()
plt.show()

## 6. Budget performance: who is actually over?

`analysis.budget_performance()` divides each category's 50-user total by the user count, then compares against that category's monthly budget.

> This is where the project's worst bug lived. Comparing the **aggregate** spend of 50 users against a **single user's** budget reported every category as roughly **+5,000%** over budget. The numbers below are only believable because the denominator and the numerator now describe the same population: one user-month.

In [ ]:
budget = analysis.budget_performance(clean)
budget

In [ ]:
fig, ax = plt.subplots(figsize=(11, 5.5))
plot = budget.sort_values("variance_pct")
colors = [config.PALETTE["rose"] if v > 0 else config.PALETTE["emerald"] for v in plot["variance_pct"]]
ax.barh(plot["category"], plot["variance_pct"], color=colors)
ax.axvline(0, color=config.PALETTE["slate"], linewidth=1)
ax.set_xlabel("Variance vs budget (%)")
ax.set_title("Budget variance per user - who is actually over")
for i, v in enumerate(plot["variance_pct"]):
    ax.text(v + (1.5 if v > 0 else -1.5), i, f"{v:+.1f}%", va="center",
            ha="left" if v > 0 else "right", fontsize=9)
plt.tight_layout()
plt.show()

worst = budget.iloc[0]
print(f"Worst overrun: {worst['category']} at {worst['variance_pct']:+.1f}% "
      f"({config.CURRENCY_SYMBOL}{worst['variance']:,.0f} per user per month above budget)")

Note what the ranking says: **Education (+94.6%) and Travel (+84.6%) are the real problems.** Food, Shopping and Groceries being 12-14% over is a rounding problem by comparison — yet those are the categories people instinctively try to cut. A tracker that only reports totals cannot tell you this.

## 7. Monthly trend, festival spikes and savings

Income is a separate file; joining it by `year_month` produces a savings rate alongside the spend trend.

In [ ]:
monthly = analysis.monthly_summary(clean, income)
monthly[[
    "year_month", "total_spend", "transactions", "mom_change_pct",
    "budget", "budget_variance_pct", "income", "savings", "savings_rate_pct",
]]

In [ ]:
fig, ax1 = plt.subplots(figsize=(12, 5))
x = pd.to_datetime(monthly["year_month"] + "-01")

ax1.bar(x, monthly["total_spend"] / 1e5, width=20, color=config.PALETTE["primary"], label="Spend")
ax1.plot(x, monthly["budget"] / 1e5, color=config.PALETTE["slate"], linewidth=2,
         linestyle="--", label="Budget (all users)")
ax1.set_ylabel("Spend (lakh)")
ax1.set_title("Monthly spend against budget, with the savings rate")
ax1.legend(loc="upper left", fontsize=9)

ax2 = ax1.twinx()
ax2.plot(x, monthly["savings_rate_pct"], color=config.PALETTE["emerald"], linewidth=2.5, marker="o", label="Savings rate")
ax2.axhline(0, color=config.PALETTE["rose"], linestyle=":", linewidth=1)
ax2.set_ylabel("Savings rate (%)")
ax2.legend(loc="upper right", fontsize=9)

plt.tight_layout()
plt.show()

neg = monthly[monthly["savings_rate_pct"] < 0]
print("Months with a negative savings rate:")
print(neg[["year_month", "total_spend", "savings_rate_pct"]].to_string(index=False))
print(f"\nPeak month {o['spend_peak_month']} vs trough {o['spend_low_month']}: "
      f"{(monthly['total_spend'].max() / monthly['total_spend'].min() - 1) * 100:.1f}% swing")

The two spike months are the whole story: they are the only months where the savings rate goes negative. A flat monthly budget cannot handle them — which is exactly why "set a pre-festival envelope" is one of the report's recommendations rather than "spend less every month".

## 8. Spending patterns: when and how

`analysis.spending_patterns()` returns weekday totals, hour buckets, the weekend split against a uniform baseline, and the payment mix.

In [ ]:
patterns = analysis.spending_patterns(clean)

print(f"Weekend share of spend : {patterns['weekend_share_pct']:.2f}%")
print(f"Uniform baseline       : {patterns['weekend_baseline_pct']:.2f}%  (2 of 7 days)")
print(f"Weekend lift           : {patterns['weekend_lift_pct']:+.2f}%")
print(f"Busiest day of week    : {patterns['busiest_day']}")
print(f"Busiest time bucket    : {patterns['busiest_bucket']}")
print(f"Top payment method     : {patterns['top_payment']}")
print()
print("Spend by weekday:")
for day, value in patterns["weekday_spend"].items():
    bar = "#" * int(value / patterns["weekday_spend"].max() * 40)
    print(f"  {day:<10} {config.CURRENCY_SYMBOL}{value / 1e5:>8,.1f} lakh  {bar}")
print()
print("Spend by time of day:")
for label, value in patterns["hour_bucket_spend"].items():
    bar = "#" * int(value / patterns["hour_bucket_spend"].max() * 40)
    print(f"  {str(label):<14} {config.CURRENCY_SYMBOL}{value / 1e5:>8,.1f} lakh  {bar}")

In [ ]:
pay = patterns["payment_spend"]
share = pay / pay.sum() * 100

fig, axes = plt.subplots(1, 2, figsize=(13, 4.6))

axes[0].pie(pay, labels=None, autopct=lambda p: f"{p:.1f}%", startangle=90,
            colors=config.CATEGORY_COLORS[: len(pay)], pctdistance=0.78,
            wedgeprops=dict(width=0.42, edgecolor="white"))
axes[0].legend(pay.index, loc="center left", bbox_to_anchor=(0.95, 0.5), fontsize=8.5)
axes[0].set_title("Payment mix by value")

weekday = patterns["weekday_spend"]
axes[1].bar(weekday.index, weekday.values / 1e5, color=config.PALETTE["secondary"])
axes[1].set_ylabel("Spend (lakh)")
axes[1].set_title("Spend by day of week")
axes[1].tick_params(axis="x", rotation=35)

plt.tight_layout()
plt.show()

print("Top rails by value:")
print(share.round(2).to_string())

> A weekend effect that does not exist is worse than no chart at all. This project originally shipped a chart implying weekend behaviour while the generated data had a weekend share of 28.3% against a 28.6% baseline — statistically nothing. The fix was to give every category its own weekend probability in the generator so the claim is supported by the data, and `test_weekend_effect_is_measurable` now fails if it ever stops being true.

## 9. Merchants: value versus frequency

Fixed monthly bills are excluded here — rent would otherwise be the "biggest merchant" in every category and drown the signal.

In [ ]:
merchants, by_category = analysis.merchant_leaders(clean)
merchants

In [ ]:
top = merchants.head(8).iloc[::-1]

fig, ax = plt.subplots(figsize=(11, 5))
ax.barh(top["merchant"] + "  (" + top["category"] + ")", top["total_spend"] / 1e5,
        color=config.PALETTE["teal"])
ax.set_xlabel("Total spend (lakh)")
ax.set_title("Top merchants by value (recurring bills excluded)")
for i, (value, txns) in enumerate(zip(top["total_spend"] / 1e5, top["transactions"])):
    ax.text(value + 0.6, i, f"{value:,.1f}L over {int(txns):,} txns", va="center", fontsize=8.5)
plt.tight_layout()
plt.show()

print("Value per transaction tells a different story from total value:\n")
top10 = merchants.head(10).copy()
top10["per_txn"] = top10["total_spend"] / top10["transactions"]
print(top10[["merchant", "category", "total_spend", "transactions", "per_txn"]].round(0).to_string(index=False))

Travel merchants dominate *value* (Airbnb, IRCTC, IndiGo) while food merchants dominate *frequency* (thousands of small orders). Same category table, two completely different budget conversations: one is about a few big decisions, the other about a daily habit.

## 10. Per-user anomaly detection

This is the product feature, and the most interesting piece of statistics in the project.

**The design:**

1. **Score each day against the user's own history**, not against the group. A ₹9,000 day is unremarkable for one user and extreme for another.
2. **Use median and MAD, not mean and standard deviation.** A single ₹50,000 booking would inflate a standard deviation enough to hide itself.
3. **Exclude recurring bills.** Paying rent on the 5th is not an anomaly, it is a schedule.
4. **Baseline only on active days.** A light spender has many near-zero days, so their median day is near zero and an ordinary ₹900 day looks like an outlier.
5. **Require z > 2.5 AND spend ≥ ₹5,000 AND at least 2 transactions.** The rupee floor stops noise, and the transaction count stops one large planned purchase from being flagged on its own.

In [ ]:
anomalies = analysis.anomaly_detection(clean)

print(f"Flagged user-days     : {anomalies['user_day_anomalies']:,}")
print(f"Users affected        : {anomalies['users_flagged']} of {clean['user_id'].nunique()}")
print(f"Alerts per user/month : {anomalies['user_day_anomalies'] / clean['user_id'].nunique() / config.N_MONTHS:.2f}")
print(f"Aggregate days flagged: {anomalies['aggregate_anomaly_days']} (the group level hides all of it)")
print()
print(f"Median daily discretionary spend : {config.CURRENCY_SYMBOL}{anomalies['median_daily_spend']:,.0f}")
print(f"MAD                              : {config.CURRENCY_SYMBOL}{anomalies['mad']:,.0f}")
print(f"Threshold                        : z > {anomalies['z_threshold']} and spend >= "
      f"{config.CURRENCY_SYMBOL}{anomalies['min_user_day_spend']:,.0f}")
print(f"Excluded as recurring            : {', '.join(anomalies['excluded_categories'])}")
print()
print(f"30-day burn rate per user        : {config.CURRENCY_SYMBOL}{anomalies['burn_rate_30d_per_user']:,.0f}")
print(f"  measured against               : {config.CURRENCY_SYMBOL}{anomalies['variable_budget_per_user']:,.0f} "
      f"variable budget  ->  {anomalies['burn_vs_variable_budget_pct']:.1f}%")

In [ ]:
top_days = anomalies["top_user_days"][
    ["day", "user_id", "spend", "user_median", "vs_own_median_pct", "z_score", "transactions"]
].copy()
top_days["day"] = pd.to_datetime(top_days["day"]).dt.date
top_days

In [ ]:
fig, ax = plt.subplots(figsize=(12, 5))
daily = anomalies["daily"].copy()
daily["date"] = pd.to_datetime(daily["date"])

ax.plot(daily["date"], daily["total_spend"] / 1e5, color=config.PALETTE["slate"],
        linewidth=0.9, alpha=0.75, label="Daily discretionary spend (all users)")
ax.plot(daily["date"], daily["rolling_avg"] / 1e5, color=config.PALETTE["primary"],
        linewidth=2.2, label=f"{config.ROLLING_WINDOW_DAYS}-day rolling average")

flagged_dates = pd.to_datetime(pd.Series(anomalies["flagged_dates"], dtype="string"), errors="coerce").dropna()
hits = daily[daily["date"].isin(flagged_dates)]
ax.scatter(hits["date"], hits["total_spend"] / 1e5, s=46, color=config.PALETTE["rose"], zorder=5,
           label=f"Day with a flagged user (n={anomalies['user_day_anomalies']})")

ax.set_ylabel("Daily spend (lakh)")
ax.set_title("Discretionary spend timeline - recurring bills excluded")
ax.legend(loc="upper right", fontsize=9)
plt.tight_layout()
plt.show()

### Why the naive version fails

The first version of this function scored the **aggregate** daily total and included rent. It flagged 927 user-days, all 50 users, and the "top anomalies" table was nothing but rent payments — a light spender whose median day is ₹600 was flagged the moment their ₹18,000 rent landed.

That is not an anomaly, it is a calendar. Three changes fixed it: exclude recurring bills, baseline each user on active days only, and add a rupee floor. The result is 657 genuinely surprising user-days at about 1.1 alerts per user per month — a rate a real user would tolerate rather than mute.

## 11. User segments

Users are generated into heavy / moderate / light archetypes, and the segmentation here is derived from the *data* independently of the generator so it is a check on the simulation as much as a summary of it.

In [ ]:
segments = analysis.segment_summary(clean)
segments

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4.6))

axes[0].pie(segments["total_spend"], labels=segments["user_segment"], autopct="%1.1f%%",
            startangle=90, colors=config.CATEGORY_COLORS[: len(segments)],
            wedgeprops=dict(width=0.45, edgecolor="white"))
axes[0].set_title("Share of total spend")

axes[1].bar(segments["user_segment"], segments["spend_per_user"] / 1e5,
            color=config.CATEGORY_COLORS[: len(segments)])
axes[1].set_ylabel("Spend per user (lakh, 12 months)")
axes[1].set_title("Spend per user by segment")
for i, v in enumerate(segments["spend_per_user"] / 1e5):
    axes[1].text(i, v, f"{v:,.1f}L", ha="center", va="bottom", fontsize=9)

plt.tight_layout()
plt.show()

heavy = segments.loc[segments["user_segment"].str.contains("Heavy"), "share_pct"].iloc[0]
heavy_users = segments.loc[segments["user_segment"].str.contains("Heavy"), "users"].iloc[0]
print(f"Heavy spenders are {heavy_users / clean['user_id'].nunique() * 100:.1f}% of users "
      f"but {heavy:.1f}% of spend.")

## 12. Key takeaways

Everything above, produced in one call — this is the same bundle the CLI saves to `analysis_summary.json`, so nothing in this notebook is hand-computed.

In [ ]:
bundle = analysis.full_analysis(clean, income)
print("full_analysis() returns:", ", ".join(bundle.keys()))
print()
print(insights.build_report(bundle)[:1800])

In [ ]:
# the ten largest single transactions, for a sense of what the extremes look like
analysis.top_transactions(clean, n=10)

## 13. Export

The notebook's job is understanding; the pipeline's job is producing the artefacts. Re-running the pipeline last guarantees the saved charts, reports and JSON all come from **one** execution rather than a mix of runs.

In [ ]:
out = data_cleaning.write_quality_report(quality)
rep = insights.write_report(bundle)

print(f"Data-quality report -> {out}")
print(f"Insight report      -> {rep}")
print()
print("To regenerate every artefact (data, 10 charts, both reports, JSON summary):")
print("    python main.py all")
print("To verify the whole project:")
print("    python -m pytest tests/ -q")